# Practice guide 04_02: dynamic analysis, real sandbox traces

**Goal.** Work the behavioural side of malware detection yourself, on **real Cuckoo traces with API names**
(MalBehavD-V1: 1,285 benign and 1,285 malware runs): turn traces into features, run the **shuffle test**,
compare two close models with **McNemar's test**, and measure what a sandbox-aware sample does to the detector.
Every `TODO` cell must be completed; the `check` cells tell you whether you got it right.

Read `guide_04_02.pdf` first. Time: about 1 hour in class, challenge at home.
Cells marked **(given)** are complete. **Environment.** `make venv` at the repository root; no GPU.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. The traces (given)

One row per run: the API calls in order (lower case). MalBehavD-V1 holds benign programs (downloads from a
software site, checked on VirusTotal) and malware (VirusTotal, 2021), all run in one Cuckoo pipeline.
We keep the first 100 calls, the window a sandbox with a call budget would record.

In [ ]:
d = mw.load_api_traces("mbd")
T = [t[:100] for t in mw.traces(d)]
y = d["malware"].to_numpy()
itr, ite = train_test_split(np.arange(len(T)), test_size=0.3, stratify=y, random_state=0)
tr_tok, te_tok, ytr, yte = [T[i] for i in itr], [T[i] for i in ite], y[itr], y[ite]
print(f"{len(T)} traces, malware {y.mean():.1%}, median {int(np.median([len(t) for t in T]))} calls")
print("first calls of one malware trace:", " ".join(T[int(np.flatnonzero(y == 1)[0])][:8]))

**Question A1.** Benign and malware traces come from *different sources*. Name one shortcut a model could
learn that would not work on a new benign program, and one check that would expose it.

## Part B. Bag of calls and $n$-grams (about 10 min)

Features: **bag of calls** (count of each API name) and **$n$-grams** (counts of consecutive pairs and triples).
Write the analyzer that turns a token list into its features; `CountVectorizer(analyzer=...)` does the counting.

In [ ]:
def ngrams(t, n_max=3):
    """Token list -> list of features: the calls themselves, then 'a>b' pairs, then 'a>b>c' triples (up to n_max)."""
    # TODO: start from the list of calls t; for n = 2 .. n_max add "a>b" (join of n consecutive calls with ">").
    raise NotImplementedError

In [ ]:
# check: tiny example, then the two models
assert ngrams(["a", "b", "c"]) == ["a", "b", "c", "a>b", "b>c", "a>b>c"]
bag = CountVectorizer(analyzer=lambda t: t)
ng = CountVectorizer(analyzer=ngrams, min_df=3)
Btr, Bte = bag.fit_transform(tr_tok), bag.transform(te_tok)
Ntr, Nte = ng.fit_transform(tr_tok), ng.transform(te_tok)
print("features: bag", Btr.shape[1], "n-grams", Ntr.shape[1])
lr_bag = LogisticRegression(max_iter=3000).fit(Btr, ytr)
lr_ng = LogisticRegression(max_iter=3000).fit(Ntr, ytr)
for name, m, X in (("bag + LR", lr_bag, Bte), ("n-gram + LR", lr_ng, Nte)):
    p = m.predict_proba(X)[:, 1]
    print(f"{name:12s} AUC {roc_auc_score(yte, p):.4f}  F1 {f1_score(yte, p > 0.5):.4f}")

**Question B1.** Why do $n$-grams give far more features than the bag, and what does a linear model on them
lose or gain?

## Part C. The shuffle test (about 15 min)

Does the $n$-gram model use order? Implement `shuffle_auc(model, vec, w, seed)`: shuffle blocks of `w` calls of
every **test** trace (`mw.block_shuffle(tokens, w, rng)`), vectorise with the *fitted* `vec`, and return the AUC.
The bag model is the control: its AUC must not move.

In [ ]:
def shuffle_auc(model, vec, w, seed=0):
    # TODO: rng = np.random.default_rng(seed); shuffled = [mw.block_shuffle(t, w, rng) for t in te_tok];
    #       return roc_auc_score(yte, model.predict_proba(vec.transform(shuffled))[:, 1])
    raise NotImplementedError

In [ ]:
a_bag = roc_auc_score(yte, lr_bag.predict_proba(Bte)[:, 1])
a_ng = roc_auc_score(yte, lr_ng.predict_proba(Nte)[:, 1])
for w in (1, 4, 100):
    s_bag = np.mean([shuffle_auc(lr_bag, bag, w, s) for s in range(3)])
    s_ng = np.mean([shuffle_auc(lr_ng, ng, w, s) for s in range(3)])
    print(f"w={w:3d}  bag {s_bag:.4f} (orig {a_bag:.4f})   n-gram {s_ng:.4f} (orig {a_ng:.4f})")
assert abs(shuffle_auc(lr_bag, bag, 1) - a_bag) < 1e-9  # the control does not move
assert shuffle_auc(lr_ng, ng, 100) == a_ng  # w = 100 leaves a 100-call trace intact

**Question C1.** The $n$-gram AUC falls less than one point. Is that "order does not matter"? What would you run
before saying so for the family task?

## Part D. McNemar's test (about 15 min)

Two models, same test traces: bag + logistic regression and bag + random forest. Count the disagreements
$b$ (A right, B wrong) and $c$ (A wrong, B right) and compute the statistic with the continuity correction,
$(|b-c|-1)^2/(b+c)$, and its p-value from `scipy.stats.chi2.sf(stat, 1)`.

In [ ]:
rf = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(Btr, ytr)
pa, pb = lr_bag.predict(Bte), rf.predict(Bte)
print(f"accuracy: LR {(pa == yte).mean():.4f}  RF {(pb == yte).mean():.4f}")


def my_mcnemar(y_true, pred_a, pred_b):
    """Return (b, c, chi2, p) with the continuity correction."""
    # TODO: ra, rb = boolean arrays "A right", "B right"; b = A right and B wrong; c = A wrong and B right;
    #       chi2 = (abs(b - c) - 1) ** 2 / (b + c); p = scipy.stats.chi2.sf(chi2, 1)
    raise NotImplementedError

In [ ]:
b, c, stat, p = my_mcnemar(yte, pa, pb)
print(f"b={b} c={c} chi2={stat:.2f} p={p:.4f}")
ref = mw.mcnemar(yte, pa, pb)
assert (b, c) == (ref["b"], ref["c"]) and abs(stat - ref["chi2"]) < 1e-9
print(f"exact binomial p = {ref['p_exact']:.4f}")

**Question D1.** With $b=7$ and $c=23$, can you say the forest is better? Why do the 700+ traces both models
get right not matter, and when must you use the exact test?

## Part E. A sandbox-aware sample (about 15 min)

An early-exit sample leaves only its first $k$ calls in the trace. Measure the recall of the bag model on the
test **malware** traces cut to $k$ calls, then add the mitigation: train on random prefixes too.

In [ ]:
mal = [t for t, lab in zip(te_tok, yte) if lab == 1]
ben = [t for t, lab in zip(te_tok, yte) if lab == 0]


def recall_at(model, vec, k):
    # TODO: share of the malware traces cut to their first k calls that the model scores above 0.5
    raise NotImplementedError

In [ ]:
for k in (100, 40, 20, 10):
    print(f"k={k:3d}  recall {recall_at(lr_bag, bag, k):.3f}")
rng = np.random.default_rng(0)
aug = [t[: int(rng.integers(3, max(4, len(t) + 1)))] for t in tr_tok for _ in range(2)]
lr_aug = LogisticRegression(max_iter=3000).fit(bag.transform(tr_tok + aug), np.r_[ytr, np.repeat(ytr, 2)])
print("with prefix augmentation:")
for k in (100, 40, 20, 10):
    print(f"k={k:3d}  recall {recall_at(lr_aug, bag, k):.3f}")
fpr = (lr_aug.predict_proba(bag.transform(ben))[:, 1] > 0.5).mean()
fpr0 = (lr_bag.predict_proba(bag.transform(ben))[:, 1] > 0.5).mean()
print(f"benign false-alarm rate: plain {fpr0:.3f}, augmented {fpr:.3f}")
assert recall_at(lr_bag, bag, 100) > recall_at(lr_bag, bag, 10)

**Question E1.** The augmented model catches more early exits. What did it cost, and what should the system
answer for a trace of 5 calls?

## Home challenges

1. **Encoder.** Train `Embedding(vocab, 32) -> GRU(64)` (Keras, JAX backend) and feed its 64-d state to a logistic
   regression. Does it beat the bag? Run the shuffle test on it with five seeds.
2. **Temporal autoencoder.** Train a GRU autoencoder on benign traces only and score the test set by
   reconstruction error. AUC without a malware label?
3. **Cascade.** Take the static model of guide 04_01 and the dynamic model of this guide; send only files with a
   static score in the grey band 0.3 to 0.7 to the dynamic model. Plot F1 against the share sent to the sandbox.